# 🚀 ATML PA3: Task 1 (LLM Fine-Tuning) — Master Kaggle Execution Notebook
**Course:** EE-5102 / CS-6304: Advanced Topics in Machine Learning (Fall 2026, LUMS)  
**Repository:** [https://github.com/moizejaz-31/ATML-PA3](https://github.com/moizejaz-31/ATML-PA3)  
**Deterministic Global Seed:** `6304`

---

### Kaggle Execution Instructions:
1. In the right-hand sidebar panel under **Notebook Settings**:
   - **Accelerator:** Set to **GPU T4 x2** (or **GPU P100**).
   - **Internet:** Toggle to **ON** *(required to download `Qwen/Qwen3-0.6B` and `MetaMathQA`)*.
2. Click **Run All** (or execute cell by cell).
3. Checkpoints, logs, and 12 high-resolution figures (300 DPI) will be saved in `/kaggle/working/ATML-PA3/task1_llm_finetuning/results/` and packaged into a downloadable zip file.


In [ ]:
# 1. Clone or Pull Repository & Install Required Packages
import os, sys

if not os.path.exists("ATML-PA3"):
    print("Cloning repository from GitHub...")
    !git clone https://github.com/moizejaz-31/ATML-PA3.git
    %cd ATML-PA3
else:
    %cd ATML-PA3
    print("Pulling latest updates from GitHub...")
    !git pull origin master

if os.path.abspath(".") not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

!pip install -q transformers datasets accelerate peft evaluate sentencepiece seaborn scikit-learn
print("\nRepository ready and dependencies installed!")


In [ ]:
# 2. Hardware & CUDA Diagnostics
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {prop.name} | Total VRAM: {prop.total_memory / 1e9:.2f} GB")
    !nvidia-smi
else:
    print("WARNING: CUDA is not active. Please set Accelerator to GPU T4 x2 in the right sidebar.")


In [ ]:
# 3. Data Preparation & GSM8K Overlap Decontamination (Seed 6304)
# Filters 40,000 MetaMathQA training samples against 1,319 GSM8K test questions.
# Samples deterministic activation probe set (32 GSM8K + 32 HellaSwag).
!python -m task1_llm_finetuning.src.data --seed 6304


In [ ]:
# 4. Pipeline Sanity Check (10-Step Smoke Test)
# Runs 10 updates on GPU to verify tokenization, forward/backward, and optimizer before launching long runs.
!python -m task1_llm_finetuning.src.train --config task1_llm_finetuning/configs/lora_r16.yaml --debug
print("\nSanity check passed! Ready for full training.")


In [ ]:
# 5. Full Training Runs (5 Configurations, 1 Epoch = 1,250 Updates each)
# Set RUN_ALL_TRAINING = True to execute all 5 training runs sequentially.
RUN_ALL_TRAINING = True

configs = [
    ("LoRA-4 (r=4, alpha=8)", "task1_llm_finetuning/configs/lora_r4.yaml"),
    ("LoRA-16 (r=16, alpha=32)", "task1_llm_finetuning/configs/lora_r16.yaml"),
    ("OPLoRA-16 (r=16, alpha=32, k=128)", "task1_llm_finetuning/configs/oplora_r16.yaml"),
    ("Full FT (Unconstrained)", "task1_llm_finetuning/configs/full_ft.yaml"),
    ("Full FT + L2-SP (lambda=1e-3)", "task1_llm_finetuning/configs/full_ft_l2sp.yaml"),
]

if RUN_ALL_TRAINING and torch.cuda.is_available():
    for name, cfg_path in configs:
        print("\n" + "="*60)
        print(f"STARTING TRAINING: {name}")
        print(f"Config: {cfg_path}")
        print("="*60)
        !python -m task1_llm_finetuning.src.train --config {cfg_path} --save_dir task1_llm_finetuning/checkpoints
else:
    print("Skipping live training. Set RUN_ALL_TRAINING = True to train.")


In [ ]:
# 6. Zero-Shot Benchmark Evaluations (GSM8K, HellaSwag, BoolQ, ARC-Easy)
!python -m task1_llm_finetuning.src.evaluate --model_name Qwen/Qwen3-0.6B --output_csv task1_llm_finetuning/results/eval_baseline.csv


In [ ]:
# 7. Render All 12 High-Resolution Publication Figures & CSVs
!PYTHONPATH=. python task1_llm_finetuning/run_task1_figures.py


In [ ]:
# 8. Display Generated Figures Inline
from IPython.display import Image, display
import glob, os

fig_files = sorted(glob.glob("task1_llm_finetuning/results/figures/*.png"))
print(f"Found {len(fig_files)} generated figures:\n")

for f in fig_files:
    print(f"--- {os.path.basename(f)} ---")
    display(Image(filename=f, width=850))


---
# 🖼️ Part 2: Task 2 — Vision-Language Models & Visual Grounding
**Architecture:** Frozen `google/siglip-base-patch16-224` + 2-layer MLP (`768 -> 1024 -> 1024`) + `Qwen/Qwen3-0.6B` + LoRA-16
**3-Stage Curriculum (4,500 Updates):**
1. Stage 1: Caption Alignment (Connector only, peak LR 1e-3, 18k COCO pairs)
2. Stage 2: Visual Instruction Tuning (50% Presence, 25% Counting, 25% Caption replay, LoRA-16)
3. Stage 3: Natural VQA Specialization (70% Natural VQAv2 consensus pairs, 10% Presence, 10% Counting, 10% Caption)
**Controls & Ablations:** Correct vs. Complementary Swap vs. Mismatched vs. No-Image, plus 196 vs 49 token compression.


In [ ]:
# 10. Task 2: SigLIP-224 Feature Extraction Diagnostics
from task2_vlm_grounding.src.feature_cache import pool_features_2x2
import torch
sample_feats = torch.randn(2, 196, 768)
pooled_feats = pool_features_2x2(sample_feats)
print(f"Original tokens: {sample_feats.shape} -> 2x2 Pooled tokens: {pooled_feats.shape}")


In [ ]:
# 11. Task 2: Curriculum Training Runner (Optional on Kaggle GPU)
RUN_TASK2_TRAINING = False # Toggle True to execute curriculum training
if RUN_TASK2_TRAINING and torch.cuda.is_available():
    print("Starting Stage 1 Alignment...")
    !python -m task2_vlm_grounding.src.train --stage 1 --config task2_vlm_grounding/configs/stage1_alignment.yaml
    print("Starting Stage 2 Visual Instruction Tuning...")
    !python -m task2_vlm_grounding.src.train --stage 2 --config task2_vlm_grounding/configs/stage2_instruction.yaml
    print("Starting Stage 3 VQA Specialization...")
    !python -m task2_vlm_grounding.src.train --stage 3 --config task2_vlm_grounding/configs/stage3_vqa.yaml
else:
    print("Curriculum checkpoints and offline evaluation tables ready.")


In [ ]:
# 12. Task 2: Control Suite Evaluation & 4,000-Pair Bootstrap CIs
!python -m task2_vlm_grounding.src.evaluate --output_dir task2_vlm_grounding/results


In [ ]:
# 13. Task 2: Render All Publication-Grade Figures & CSV Tables
!PYTHONPATH=. python task2_vlm_grounding/run_task2_figures.py


In [ ]:
# 14. Task 2: Display Generated Figures Inline
from IPython.display import Image, display
import glob, os
t2_figs = sorted(glob.glob("task2_vlm_grounding/results/figures/*.png"))
print(f"Found {len(t2_figs)} Task 2 figures:\n")
for f in t2_figs:
    print(f"--- {os.path.basename(f)} ---")
    display(Image(filename=f, width=850))


In [ ]:
# 15. Create Master Archive (Task 1 + Task 2 Results, CSVs, Figures)
!zip -r /kaggle/working/atml_pa3_master_artifacts.zip task1_llm_finetuning/results task2_vlm_grounding/results
print("Master archive created: /kaggle/working/atml_pa3_master_artifacts.zip")
